# Config 1: Gemma, CPU-only, zero-shot (Colab)

The floor of the three-config comparison: an off-the-shelf open-weight
model, **no fine-tuning and no LoRA adapter**, no tool access, running
entirely on ordinary CPU -- the air-gapped baseline. `config2-qlora-gpu/`
QLoRA-fine-tunes the *same* base model on a GPU;
`config3-frontier-skills/` gives a frontier model SAS-metadata tool
access. See the repo root's `PLAN.md` for the full three-way framing.

This notebook runs **one path only**: Hugging Face `transformers` loading
`google/gemma-3-4b-it` -- the exact weights config2 fine-tunes -- on a
free Colab **CPU** runtime.

| | this notebook |
|---|---|
| where | Colab, free tier |
| accelerator | **none** -- CPU only, enforced in section 2 |
| model source | `transformers` downloads `google/gemma-3-4b-it` from the Hub |
| weights | HF safetensors, bfloat16 |
| adapter | **none** -- base weights as published, config2's `sasdoc-lora` is never loaded |
| base-model parity with config2 | exact (same repo id, same revision) |

Because the model and the weights are identical to config2's base, the
config1-vs-config2 row in the results table isolates **fine-tuning**, and
the config1-vs-config3 row isolates **tool access** -- neither is
contaminated by a different model or a different serving stack.

**Runtime -> Change runtime type -> CPU (or "None").** No accelerator is
needed, and needing none is the entire point of config1 versus config2's
T4 requirement. If a GPU runtime is attached anyway, section 2 hides it
so the numbers stay honest CPU numbers.

> `document_sas.py` also carries an `--backend ollama` path for a box
> with a local Ollama/GGUF server. This notebook does not use it and
> never contacts `localhost` -- see `SETUP.md` if you want that path.

Run top to bottom: get the repo -> install deps -> force CPU + check RAM
-> authenticate to the Hub -> generate documentation -> score -> save the
outputs off Colab -> *(optional)* push results to SAS via ODA.

## 0. Get the whole repo (Colab)

Pull the **whole** repo, not just this folder: sections 4-5 need
`../eval-programs/` (the 20 held-out programs + gold) and `../results/`
(`score.py`, `run_eval.py`) as siblings of this folder. Pulling only
`config1-gemma-cpu/` leaves every `../eval-programs/...` and
`../results/...` path below failing with "No such file or directory".

Off Colab this cell is a no-op beyond confirming where it is running, so
the notebook is the same file in both places.

In [1]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/patrickjlong1/sas-llm-paper.git"
CLONE_DIR = "/content/sas-llm-paper"

if IN_COLAB:
    if not os.path.isdir(CLONE_DIR):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, CLONE_DIR],
                       check=True)
    os.chdir(os.path.join(CLONE_DIR, "config1-gemma-cpu"))

REPO_ROOT = os.path.dirname(os.getcwd())
print("running on Colab:", IN_COLAB)
print("cwd:            ", os.getcwd())
for need in ("../eval-programs/programs", "../eval-programs/gold", "../results"):
    print("  %-28s %s" % (need, "OK" if os.path.isdir(need) else "MISSING"))

running on Colab: True
cwd:             /content/sas-llm-paper/config1-gemma-cpu
  ../eval-programs/programs    OK
  ../eval-programs/gold        OK
  ../results                   OK


## 1. Python dependencies

`transformers` + `torch` + `accelerate` + `huggingface_hub` -- that is the
whole runtime for this config. Colab already ships `torch`, so this is
mostly a `transformers` version bump; budget a couple of minutes.

No `bitsandbytes` (4-bit quantization is config2's GPU path -- its kernels
are CUDA-only) and no `peft` (no adapter is loaded here). `requests` comes
along because `document_sas.py` imports it at module level for its unused
`ollama` backend; nothing in this notebook opens an HTTP connection to a
local server.

The optional ODA push in section 7 additionally needs `saspy` + `pandas`;
that section installs them itself.

In [2]:
!pip -q install -r requirements.txt

## 2. CPU only -- make it so, and check the RAM

Two things this cell guarantees, both of which the paper depends on:

1. **CPU only.** `hf_backend.py` already passes `device_map="cpu"`, but
   emptying `CUDA_VISIBLE_DEVICES` means even an accidentally-attached
   GPU runtime cannot contribute -- so a timing in section 4 is a CPU
   timing, full stop. It must run **before** `torch` is first imported in
   this kernel; if you attached a GPU after running it, use Runtime ->
   Restart session and re-run from the top.
2. **Enough RAM.** `google/gemma-3-4b-it` in bfloat16 is ~8.6 GB of
   weights. Free Colab's CPU runtime has ~12.7 GB, so it fits, but not
   with much room -- and a `^C`-less kernel death partway through a batch
   run is what running out looks like.

In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = ""   # before torch is imported

import torch

print("torch:", torch.__version__)
print("CUDA visible to torch:", torch.cuda.is_available(), "(must be False)")

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
    print("system RAM: %.1f GB" % ram_gb)
    if ram_gb < 10:
        print("\nWARNING: under 10 GB. gemma-3-4b-it in bfloat16 needs ~8.6 GB for "
              "weights alone and will likely OOM the kernel mid-run. See the "
              "smaller-model note in section 3.")
except ImportError:
    pass

torch: 2.11.0+cpu
CUDA visible to torch: False (must be False)
system RAM: 13.6 GB


## 3. Run settings

`google/gemma-3-4b-it` is **gated**: accept the license on its Hugging
Face page while logged in, then run the auth cell below. Prefer Colab's
Secrets panel (padlock icon, left sidebar) with an `HF_TOKEN` secret --
`login()` picks that up and only prompts if it isn't set. This is the
same gate config2 goes through for the same weights.

`REVISION` is pinned to `main` to match config2's default; set both to the
same commit sha if you want the comparison reproducible against a moving
Hub repo.

In [4]:
import json

MODEL = "google/gemma-3-4b-it"   # exactly what config2 fine-tunes -- keep them equal
REVISION = "main"                # Hub revision; a sha here makes the run reproducible
DTYPE = "auto"                   # auto == bfloat16; see document_sas.py --dtype

RUN = "config1-gemma-cpu"
OUT = "../results/preds/" + RUN
CAT = "../results/catalog/" + RUN
PREFIX = "../results/outputs/" + RUN        # run_eval.py --out-prefix
SCORES = PREFIX + ".scores.jsonl"

# Set as env vars as well as Python names so the `!` cells below read the
# same way if you paste them into a terminal. (Keep every shell reference a
# bare `$NAME` followed by whitespace: IPython's `$` expansion swallows a
# following dot as attribute access, so `$PREFIX.scores.jsonl` would break --
# hence the separate SCORES variable.)
os.environ.update(MODEL=MODEL, REVISION=REVISION, DTYPE=DTYPE, OUT=OUT, CAT=CAT,
                  RUN=RUN, PREFIX=PREFIX, SCORES=SCORES)
print("model=%s  revision=%s  dtype=%s" % (MODEL, REVISION, DTYPE))
print("predictions -> %s" % OUT)
print("\nNOTE: 4B in bfloat16 on a free Colab CPU is memory-feasible but slow "
      "(PyTorch has no fast bf16 CPU kernels without AVX512-BF16/AMX). If the "
      "timing in section 4 is impractical, the documented faster option is "
      "MODEL='google/gemma-3-1b-it' with DTYPE='float32' -- report it as such, "
      "since it breaks exact base-model parity with config2 and so weakens what "
      "the config1-vs-config2 row can claim.")

model=google/gemma-3-4b-it  revision=main  dtype=auto
predictions -> ../results/preds/config1-gemma-cpu

NOTE: 4B in bfloat16 on a free Colab CPU is memory-feasible but slow (PyTorch has no fast bf16 CPU kernels without AVX512-BF16/AMX). If the timing in section 4 is impractical, the documented faster option is MODEL='google/gemma-3-1b-it' with DTYPE='float32' -- report it as such, since it breaks exact base-model parity with config2 and so weakens what the config1-vs-config2 row can claim.


In [5]:
from huggingface_hub import login
login()   # no-op prompt-wise if the HF_TOKEN Colab secret is set

## 4. Run it

Zero-shot, base weights: no fine-tuning and no adapter (that's config2).
`transformers` has no equivalent of a JSON-mode decoding constraint, so
the model is free to emit prose around or instead of the JSON;
`schema.py`'s `validate()` checks the CONTENT shape regardless, and
`results/score.py` reports schema validity as its own metric rather than
hiding those failures.

Each program writes `<name>.pred.json` (the structured dictionary),
`<name>.meta.json` (timing/cost, in the shape `results/run_eval.py`
expects), `<name>.raw.txt` (the raw response, kept even on a parse
failure), and `<name>.guardrail.json` (hallucination check against
`extract.py`'s static source scan). Every run also upserts into a local,
offline JSONL catalog (`catalog.py`) that section 7 can push to SAS.

**One program first** -- both to see the output shape and to get a real
per-program time before committing to all 20. This first call also pays
the one-time weight download (~8.6 GB) and load; every later program
reuses the cached model in-process. The cell after it does the arithmetic
for you.

In [6]:
!python3 document_sas.py ../eval-programs/programs/prog900_estab.sas \
    --backend hf --model $MODEL --revision $REVISION --dtype $DTYPE \
    --out $OUT --catalog $CAT

loading google/gemma-3-4b-it (revision=main) as bfloat16 on CPU -- one-time cost per run, reused for every program after this...
config.json: 100% 855/855 [00:00<00:00, 2.53MB/s]
tokenizer_config.json: 100% 1.16M/1.16M [00:00<00:00, 36.8MB/s]

tokenizer.json: downloading bytes:  21% 7.16M/33.4M [00:00<00:02, 13.0MB/s, 4.43kB/s  ]
tokenizer.json: downloading bytes: 100% 8.83M/8.83M [00:00<00:00, 11.2MB/s,  863kB/s  ]
tokenizer.json: reconstructing file: 100% 33.4M/33.4M [00:00<00:00, 42.2MB/s, 3.27MB/s  ]
added_tokens.json: 100% 35.0/35.0 [00:00<00:00, 150kB/s]
special_tokens_map.json: 100% 662/662 [00:00<00:00, 2.72MB/s]
model.safetensors.index.json: 100% 90.6k/90.6k [00:00<00:00, 93.9MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/8.60G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   3% 268M/8.60G [00:02<01:05, 127MB/s, 49.8MB/s  ]
Reconstr

In [7]:
# How long would all 20 take at the speed just measured? Colab free
# sessions idle-disconnect after ~90 minutes and cap around 12 hours, so
# decide here rather than discovering it eight hours in.
meta_path = os.path.join(OUT, "prog900_estab.meta.json")
if os.path.exists(meta_path):
    secs = json.load(open(meta_path))["elapsed_sec"]
    print("measured: %.1f s for 1 program" % secs)
    print("estimated: %.1f min for all 20 (download + model load already paid once)"
          % (secs * 20 / 60.0))
    if secs * 20 > 3 * 3600:
        print("\n>3h for the full set. Options: run the batch cell with --limit N "
              "and report a partial run AS partial, switch to the faster "
              "1B/float32 combination in section 3, or run the full set "
              "somewhere without a session cap.")
else:
    print("run the cell above first")

measured: 2619.8 s for 1 program
estimated: 873.3 min for all 20 (download + model load already paid once)

>3h for the full set. Options: run the batch cell with --limit N and report a partial run AS partial, switch to the faster 1B/float32 combination in section 3, or run the full set somewhere without a session cap.


Then the whole eval set -- one program at a time, in sequence (no GPU, no
request batching). A per-file failure (bad file, timeout, OOM) is logged
and skipped rather than aborting the run.

Add `--limit N` for a partial smoke run. A partial run is **not** a
result: `score.py` scores against all 20 gold files and counts every
missing prediction as a program that produced no output, so a 5-program
run scores like a 20-program run that failed 15 times.

In [ ]:
!python3 document_sas.py --dir ../eval-programs/programs \
    --backend hf --model $MODEL --revision $REVISION --dtype $DTYPE \
    --out $OUT --catalog $CAT

batch: 20 file(s) in ../eval-programs/programs

[1/20] ../eval-programs/programs/prog900_estab.sas
loading google/gemma-3-4b-it (revision=main) as bfloat16 on CPU -- one-time cost per run, reused for every program after this...
Loading weights: 100% 883/883 [00:01<00:00, 842.64it/s]
model: google/gemma-3-4b-it | 2590.2s
wrote ../results/preds/config1-gemma-cpu/prog900_estab.pred.json (raw kept at ../results/preds/config1-gemma-cpu/prog900_estab.raw.txt)
wrote ../results/preds/config1-gemma-cpu/prog900_estab.meta.json
wrote ../results/preds/config1-gemma-cpu/prog900_estab.guardrail.json
no guardrail flags -- all claimed identifiers were found in the source.
catalog: ../results/catalog/config1-gemma-cpu -> 4 macro row(s), 10 dictionary row(s) (prog900_estab)

[2/20] ../eval-programs/programs/prog901_hhold.sas


## 5. Score this run

`results/run_eval.py` wraps `score.py` into the `.scores.jsonl` the plan's
results table reads, and stamps a `.provenance.json` sidecar recording
exactly which eval corpus was scored -- `--table` marks a row **STALE**
rather than printing numbers that no longer refer to the corpus on disk.
(That check exists because it already went wrong once; see
`../results/outputs/stale-2026-09-17/README.md`.)

`--run-judge` is deliberately left off: the default judge is a local
Ollama model, which Colab doesn't have. Run the judge later on a box that
does; until then the table prints `not run` for Description score rather
than a made-up number.

In [ ]:
!python3 ../results/run_eval.py --config $RUN --pred-dir $OUT --out-prefix $PREFIX

In [ ]:
!python3 ../results/run_eval.py --table \
    --scores $SCORES --meta-dir $OUT \
    --label "Config 1: Gemma CPU zero-shot ($MODEL)"

## 6. Get the outputs off Colab

Colab's local disk does **not** survive a runtime recycle, and neither do
the downloaded weights. Unlike config2 there's no trained artifact to
lose here -- but the predictions, catalog and scores ARE the run, and
re-creating them costs the whole generation time again.

Skip this cell off Colab.

In [ ]:
if IN_COLAB:
    from google.colab import files
    import shutil
    for src, name in ((OUT, RUN + "-preds"),
                      (CAT, RUN + "-catalog"),
                      ("../results/outputs", RUN + "-scores")):
        if os.path.isdir(src):
            shutil.make_archive("/content/" + name, "zip", src)
            files.download("/content/%s.zip" % name)
else:
    print("not on Colab -- outputs are already on local disk")

## 7. (Optional) Push the catalog into SAS via ODA

Only needed to materialize the generated dictionary as real SAS datasets
(`push_to_oda.py`). Config1 has no ground-truth SAS access **by design**
-- that asymmetry is config3's job and part of the finding -- so nothing
here feeds back into config1's own output. Skip this section entirely if
you only want the local JSON/catalog output.

**Get a free ODA account** first if you don't have one (SAS OnDemand for
Academics signup -- no cost, academic/non-commercial use).

**Credentials.** On Colab, use the Secrets panel (padlock icon) with
`ODA_USER` / `ODA_PASS` secrets -- the cell below reads them from there
and writes `~/.authinfo` into the ephemeral session, prompting only if
the secrets aren't set, so the password never becomes part of the saved
notebook. On your own machine, create `~/.authinfo` yourself, in a
terminal:

```bash
echo "oda user YOUR_ODA_EMAIL password YOUR_ODA_PASSWORD" >> ~/.authinfo
chmod 600 ~/.authinfo
```

**Java.** `config/sascfg_personal.py` prefers the portable JRE at the repo
root (`../jre/`) and falls back to whatever `java` is on `PATH`. `jre/` is
136 MB and gitignored, so a fresh clone (i.e. Colab) does **not** have it
-- install a JDK there, which the cell below does.

**Region.** `config/sascfg_personal.py`'s `iomhost` list is already filled
in for a US-region/usw2 account; see `SETUP.md` for the Europe and Asia
Pacific host names.

In [ ]:
import shutil

if IN_COLAB:
    !apt-get -qq install default-jdk > /dev/null
    !pip -q install saspy pandas
print("java:", shutil.which("java") or "NOT ON PATH (repo jre/ will be used if present)")

In [ ]:
import getpass

authinfo = os.path.expanduser("~/.authinfo")
if os.path.exists(authinfo):
    print("~/.authinfo found -- leaving it alone")
else:
    user = password = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            user, password = userdata.get("ODA_USER"), userdata.get("ODA_PASS")
        except Exception:
            pass
    user = user or input("ODA email: ")
    password = password or getpass.getpass("ODA password: ")
    with open(authinfo, "w") as fh:
        fh.write("oda user %s password %s\n" % (user, password))
    os.chmod(authinfo, 0o600)
    del password
    print("wrote", authinfo, "(chmod 600)")

In [ ]:
# Writes PROGRAM_SUMMARY, MACRO_PARAMS, DATA_DICTIONARY into your SASUSER
# library (SAS's auto-assigned, persistent-across-sessions library -- no
# LIBNAME statement or path to know). Pass --libname/--libpath only for a
# different, custom-path library.
#
# On Colab saspy lands in the notebook kernel itself, so sys.executable is
# the right interpreter; the fallback covers a local box that keeps saspy in
# a separate venv.
VENV_PY = "/internal/venvs/main/bin/python3"
PY = VENV_PY if (not IN_COLAB and os.path.exists(VENV_PY)) else sys.executable
os.environ["PY"] = PY
print("using interpreter:", PY)

In [ ]:
!$PY push_to_oda.py --catalog $CAT

## Why zero-shot, not few-shot or fine-tuned

Tested in-context learning (showing the model 1-2 example SAS->JSON
pairs) on the small Gemma models and it made output WORSE, not better:
with even one exemplar in context, a 1B-class model lost coherence and
fabricated an entirely fictional SAS program instead of documenting the
real one. That's a genuine capability ceiling, not a prompt bug -- if you
want few-shot or fine-tuned quality, that's config2
(`../config2-qlora-gpu/`), which is also where any LoRA adapter belongs.
Nothing in this notebook loads `../adapters/sasdoc-lora`; if it did, the
config1 row would stop being a baseline.

## Known limitations (report these honestly in the results)

- **No JSON-mode decoding.** `transformers` has no equivalent of Ollama's
  `format: "json"`, so schema-invalid output is common: missing keys,
  wrong types, a list entry that's a bare string instead of an object,
  or prose wrapped around the JSON. `document_sas.py` does NOT crash on
  this -- it records `schema_valid: false` and moves on, exactly so
  `results/score.py` can report schema validity as its own honest metric
  rather than one bad program aborting a batch run.
- Even when schema-valid, meanings/types are genuinely best-guess and
  were observed wrong on toy synthetic input (e.g. calling a sampling
  weight "Work Time"). The guardrail only catches INVENTED names, never
  wrong MEANINGS -- read every entry, don't just trust an absence of
  flags.
- `extract.py`'s regex scan is a heuristic on arbitrary real SAS syntax
  -- it can miss real identifiers written in forms it doesn't anticipate,
  which under-flags (reports clean when it isn't).
- **Report the hardware.** "CPU" is not one thing: Colab's free CPU
  runtimes vary in core count and in whether they have AVX512-BF16, which
  changes bf16 throughput by a lot. `.meta.json` records the dtype;
  record the machine yourself (PLAN.md section 3, "Also record").
- If you fall back to `gemma-3-1b-it`/float32 for speed, that is a
  **different row**, not a faster version of this one -- it is no longer
  config2's base model, so it cannot carry the config1-vs-config2
  fine-tuning claim. Write predictions to a different `RUN` name if you
  want to keep both.
- A `--limit`ed run is a smoke test, not a result. `score.py` counts every
  un-predicted program as a failure, so a partial run's numbers are not
  comparable to a full one's.